# LayoutExtractor (Docling) Test Notebook

This notebook demonstrates and tests the `LayoutExtractor` strategy (Strategy B)
which uses **Docling** for layout-aware extraction:

- Multi-column text handling
- Table structure preservation
- Figures/images with captions
- Normalisation into the shared `ExtractedDocument` schema.


In [ ]:
## Setup and Imports

import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path().absolute().parent))

from src.agents.triage import TriageAgent
from src.strategies.layout_aware import LayoutExtractor
from src.models.document_profile import DocumentProfile


In [ ]:
## Select Test Document

# Reuse the same corpus paths as the fast text notebook
TEST_DOCS = {
    "class_a": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_a\CBE_ANNUAL_REPORT_2023-24.pdf"),
    "class_b": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_b\Annual_Report_JUNE-2023.pdf"),
    "class_c": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_c\fta_performance_survey_final_report_2022.pdf"),
    "class_d": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_d\tax_expenditure_ethiopia_2021_22.pdf"),
}

selected_doc = "class_a"  # change to b/c/d to test other classes
pdf_path = TEST_DOCS[selected_doc]

if not pdf_path.exists():
    print(f"⚠️ Document not found: {pdf_path}")
else:
    print(f"✓ Selected document: {pdf_path.name}")
    print(f"  Path: {pdf_path}")
    print(f"  Size: {pdf_path.stat().st_size / 1024 / 1024:.2f} MB")


In [ ]:
## Step 1: Triage and Profile

profiles_dir = Path(".refinery/profiles")
profiles_dir.mkdir(parents=True, exist_ok=True)
triage = TriageAgent(profiles_dir=profiles_dir)

print("🔍 Running Triage Agent...")
profile = triage.classify_document(pdf_path)

print("\n📋 Document Profile (summary):")
print(f"  - Document ID: {profile.doc_id}")
print(f"  - Origin: {profile.origin_type}")
print(f"  - Layout: {profile.layout_complexity}")
print(f"  - Domain: {profile.domain_hint}")
print(f"  - Estimated Cost: {profile.estimated_cost}")
print(f"  - Language: {profile.language} (confidence: {profile.language_confidence:.2f})")
print(f"  - Pages: {profile.metadata.page_count}")
print(f"  - Size: {profile.metadata.size_bytes / 1024 / 1024:.2f} MB")


In [ ]:
## Step 2: Initialise LayoutExtractor

try:
    layout_extractor = LayoutExtractor()
    print("✓ Initialised LayoutExtractor (Docling)")
except ImportError as e:
    print("✗ Could not initialise LayoutExtractor:", e)
    print("  Make sure docling is installed: pip install docling")
    raise

print("\n🔍 Checking if LayoutExtractor can handle this profile...")
can_handle = layout_extractor.can_handle(profile)
print(f"  - Can Handle: {can_handle}")


In [ ]:
## Step 3: Confidence and Cost

print("📊 Calculating layout-aware confidence score...")
layout_conf = layout_extractor.confidence_score(str(pdf_path))
print(f"  - Confidence: {layout_conf:.4f} ({layout_conf*100:.1f}%)")

print("\n💰 Estimating layout-aware extraction cost...")
layout_cost = layout_extractor.cost_estimate(str(pdf_path))
print(f"  - Total Cost: ${layout_cost['total_cost_usd']:.4f}")
print(f"  - Cost per Page: ${layout_cost['cost_per_page']:.4f}")


In [ ]:
## Step 4: Run Layout-Aware Extraction

print("📄 Extracting with LayoutExtractor (Docling)...")

import time
start_time = time.time()

extracted = layout_extractor.extract(str(pdf_path))

elapsed = time.time() - start_time
print(f"\n✓ Extraction completed in {elapsed:.2f} seconds")
print("\n📊 Extraction Summary (Docling → ExtractedDocument):")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")
print(f"  - Reading Order indices: {len(extracted.reading_order)}")


In [ ]:
## Step 5: Inspect Sample Tables and Figures

print("📊 Sample Tables (first 3):")
print("=" * 80)
for i, table in enumerate(extracted.tables[:3], start=1):
    print(f"\n[Table {i}] Page {table.page_num}")
    print(f"  Headers: {table.headers}")
    if table.rows:
        print(f"  First row: {table.rows[0]}")

print("\n🖼️ Sample Figures (first 3):")
print("=" * 80)
for i, fig in enumerate(extracted.figures[:3], start=1):
    print(f"\n[Figure {i}] Page {fig.page_num}")
    print(f"  Caption: {fig.caption or '(none)'}")
    print(f"  BBox: ({fig.bbox.x0:.1f}, {fig.bbox.y0:.1f}) → ({fig.bbox.x1:.1f}, {fig.bbox.y1:.1f})")


In [ ]:
## Step 6: Summary

print("✅ LayoutExtractor (Docling) Test Summary:")
print("=" * 80)
print(f"\nDocument: {pdf_path.name}")
print("\nProfile:")
print(f"  - Origin: {profile.origin_type}")
print(f"  - Layout: {profile.layout_complexity}")
print(f"  - Domain: {profile.domain_hint}")
print("\nStrategy:")
print(f"  - Strategy: {layout_extractor.name}")
print(f"  - Can Handle: {can_handle}")
print(f"  - Confidence: {layout_conf:.2%}")
print(f"  - Estimated Cost: ${layout_cost['total_cost_usd']:.4f}")
print("\nExtraction Results:")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")
